# Asking a table questions in plain words

Built from DSCI 532 lectures 6 and 7 (querychat, the taxi apps, the RAG demo).
Data: NYC TLC yellow taxi trips, January 2024.

In [1]:
import os
from pathlib import Path

import chatlas as ctl
import duckdb
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from querychat import QueryChat
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# API keys: from the environment, or from ../.env (copy .env.example to .env and add your own keys); never printed
env = Path("../.env")
for line in env.read_text().splitlines() if env.exists() else []:
    name, _, value = line.partition("=")
    if name.strip() in ("GROQ_API_KEY", "OPENAI_API_KEY") and value.strip() and not os.environ.get(name.strip()):
        os.environ[name.strip()] = value.strip()
print("keys loaded:", [k for k in ("GROQ_API_KEY", "OPENAI_API_KEY") if os.environ.get(k)])

keys loaded: ['GROQ_API_KEY', 'OPENAI_API_KEY']


## 1. The problem

An analyst has a month of taxi trips and asks a question in plain words, not in SQL.

In [3]:
taxi = duckdb.sql("SELECT * FROM '../data/taxi_2024-01.parquet'").df()
print(f"Loaded {len(taxi):,} rows × {len(taxi.columns)} columns")
taxi.head()

Loaded 2,964,624 rows × 19 columns


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [4]:
question = "What is the average tip by payment type?"

Reference query, stored for the closing comparison:

In [5]:
reference = duckdb.sql("""
    SELECT payment_type, COUNT(*) AS trips, ROUND(AVG(tip_amount), 2) AS avg_tip
    FROM taxi
    GROUP BY payment_type
    ORDER BY payment_type
""").df()

## 2. What the model sees

`QueryChat(taxi, "taxi")` builds the system prompt, registers the tools, and talks to the model through chatlas.

In [6]:
qc = QueryChat(
    taxi,
    "taxi",
    client=ctl.ChatGroq(model="qwen/qwen3.8-27b"),
    # client=ctl.ChatOpenAI(model="gpt-5.4-mini"),
    tools=("update", "query"),
)

In [7]:
print(qc.system_prompt)

You are a data dashboard chatbot that operates in a sidebar interface. Your role is to help users interact with their data through filtering, sorting, and answering questions.

You have access to a DuckDB SQL database with the following tables:

<tables>
- taxi
</tables>

Always call `querychat_get_schema` before writing SQL against any table you haven't retrieved schema for in this conversation. Do not infer column names from table names, variable names, or the system prompt alone — verify the actual schema first, then write the query.

For security reasons, you may only query this specific table.

## SQL Query Guidelines

When writing SQL queries to interact with the database, please adhere to the following guidelines to ensure compatibility and correctness.

### Structural Rules

**No trailing semicolons**
Never end your query with a semicolon (`;`). The parent query needs to continue after your subquery closes.

**Single statement only**
Return exactly one `SELECT` statement. Do no

The tools the model is offered (it can only **ask** for a call; our Python code runs it):

In [8]:
for tool in qc.client().get_tools():
    print(tool.name)

querychat_get_schema
querychat_update_dashboard
querychat_reset_dashboard
querychat_query


### Schema sent by the tool

`querychat_get_schema` returns column names, types, ranges and small category lists. It sends no data rows by default. Rows reach the model only through a query tool.

In [9]:
schema_tool = [t for t in qc.client().get_tools() if t.name == "querychat_get_schema"][0]
schema_result = schema_tool.func(table_name="taxi")
print(schema_result.value)

Table: taxi
Columns:
- VendorID (INTEGER)
  Range: 1 to 6
- tpep_pickup_datetime (TIMESTAMP)
  Range: 2002-12-31 22:59:39 to 2024-02-01 00:01:15
- tpep_dropoff_datetime (TIMESTAMP)
  Range: 2002-12-31 23:05:41 to 2024-02-02 13:56:52
- passenger_count (INTEGER)
  Range: 0 to 9
- trip_distance (FLOAT)
  Range: 0.0 to 312722.3
- RatecodeID (INTEGER)
  Range: 1 to 99
- store_and_fwd_flag (TEXT)
  Categorical values: 'N', 'Y'
- PULocationID (INTEGER)
  Range: 1 to 265
- DOLocationID (INTEGER)
  Range: 1 to 265
- payment_type (INTEGER)
  Range: 0 to 4
- fare_amount (FLOAT)
  Range: -899.0 to 5000.0
- extra (FLOAT)
  Range: -7.5 to 14.25
- mta_tax (FLOAT)
  Range: -0.5 to 4.0
- tip_amount (FLOAT)
  Range: -80.0 to 428.0
- tolls_amount (FLOAT)
  Range: -80.0 to 115.92
- improvement_surcharge (FLOAT)
  Range: -1.0 to 1.0
- total_amount (FLOAT)
  Range: -900.0 to 5000.0
- congestion_surcharge (FLOAT)
  Range: -2.5 to 2.5
- Airport_fee (FLOAT)
  Range: -1.75 to 1.75


### data_description & extra_instructions

These go into the **system prompt** — they change how the LLM reasons about data and formats its answers.

In [12]:
qc_custom = QueryChat(
    taxi,
    "taxi",
    client=ctl.ChatGroq(model="qwen/qwen3.8-27b"),
    tools=("update", "query"),
    data_description=(
        "NYC yellow taxi trips from January 2024, one row per trip, from the NYC TLC trip records. "
        "Includes pickup and dropoff times and zones, distance, fare, tip and payment type."
    ),
    extra_instructions=(
        "Format dollar amounts with $ and two decimals. "
        "Keep responses to 2-3 sentences."
    ),
)

# See what changed in the prompt
prompt = qc_custom.system_prompt
for tag in ["data_description", "Additional Instructions"]:
    idx = prompt.find(tag)
    if idx >= 0:
        print(prompt[max(0, idx - 20):idx + 300])
        print("...\n")


- taxi
</tables>

<data_description>
NYC yellow taxi trips from January 2024, one row per trip, from the NYC TLC trip records. Includes pickup and dropoff times and zones, distance, fare, tip and payment type.
</data_description>

Always call `querychat_get_schema` before writing SQL against any table you haven't retr
...

relevant subset

## Additional Instructions

Format dollar amounts with $ and two decimals. Keep responses to 2-3 sentences.

...



## 3. One question end to end

The `on_tool_request` and `on_tool_result` callbacks show each tool request, the result returned to the model and the display sent to the screen.

In [13]:
client = qc.client()

step = [0]


# chatlas calls these two functions back on every tool request and result.
# Inline alternative without callbacks: run the chat, then read client.get_turns() (cell after next).
def show_request(req):
    step[0] += 1
    print(f"\n── Step {step[0]}: LLM requests tool ─────────────────")
    print(f"   Tool:      {req.name}")
    print(f"   Arguments: {req.arguments}")


def show_result(res):
    step[0] += 1
    print(f"\n── Step {step[0]}: Tool returns result ──────────────────")
    print(f"   Sent back to LLM: {str(res.value)[:300]}")
    display = (res.extra or {}).get("display")
    print(f"   Shown on screen:  {str(getattr(display, 'markdown', None) or display)[:300]}")


client.on_tool_request(show_request)
client.on_tool_result(show_result);

In [14]:
step[0] = 0
print("── Step 0: User sends message ──────────────────────")
print(f'   "{question}"')

response = client.chat(question, echo="none")

step[0] += 1
print(f"\n── Step {step[0]}: LLM responds to user ─────────────────")
print(f"   {response}")

── Step 0: User sends message ──────────────────────
   "What is the average tip by payment type?"



── Step 1: LLM requests tool ─────────────────
   Tool:      querychat_get_schema
   Arguments: {'table_name': 'taxi'}



── Step 2: Tool returns result ──────────────────
   Sent back to LLM: Table: taxi
Columns:
- VendorID (INTEGER)
  Range: 1 to 6
- tpep_pickup_datetime (TIMESTAMP)
  Range: 2002-12-31 22:59:39 to 2024-02-01 00:01:15
- tpep_dropoff_datetime (TIMESTAMP)
  Range: 2002-12-31 23:05:41 to 2024-02-02 13:56:52
- passenger_count (INTEGER)
  Range: 0 to 9
- trip_distance (FLOAT)
   Shown on screen:  title=None label='taxi' value_preview='19 columns' icon=None html=<div class="table-responsive">
  <table class="table table-sm table-hover align-middle mb-0">
    <thead class="table-light">
      <tr>
        <th scope="col">Column</th>
        <th scope="col">Type</th>
        <th scope="col">Ran



── Step 3: LLM requests tool ─────────────────
   Tool:      querychat_query
   Arguments: {'_intent': 'Calculate average tip amount for each payment type', 'collapsed': True, 'query': 'SELECT\n  payment_type,\n  AVG(tip_amount) AS avg_tip\nFROM\n  taxi\nGROUP BY\n  payment_type'}



── Step 4: Tool returns result ──────────────────
   Sent back to LLM: [{'payment_type': 1, 'avg_tip': 4.169670627491175}, {'payment_type': 4, 'avg_tip': 0.04212511795487692}, {'payment_type': 3, 'avg_tip': 0.014559881614532833}, {'payment_type': 2, 'avg_tip': 0.0022960169948837753}, {'payment_type': 0, 'avg_tip': 1.5459567500463327}]
   Shown on screen:  ```sql
SELECT
  payment_type,
  AVG(tip_amount) AS avg_tip
FROM
  taxi
GROUP BY
  payment_type
```

<div id="hgjtrlfean" style="padding-left:0px;padding-right:0px;padding-top:10px;padding-bottom:10px;overflow-x:auto;overflow-y:auto;width:auto;height:auto;">
<style>
#hgjtrlfean table {
          font



── Step 5: LLM responds to user ─────────────────
   To answer this, I first need to understand the structure of the data. I will start by getting the schema for the `taxi` table, which should contain information about payment types and tips.

I can see that `payment_type` is an integer and `tip_amount` is a float in the `taxi` table. I will now calculate the average `tip_amount` grouped by `payment_type`.

Here is the average tip amount by payment type:

| payment_type | avg_tip |
| :----------- | :------ |
| 0            | $1.55   |
| 1            | $4.17   |
| 2            | $0.00   |
| 3            | $0.01   |
| 4            | $0.04   |

Payments of type 1 (likely credit cards) have the highest average tip at **$4.17**.

<span class="suggestion">Filter the data to show only rides with payment type 1 (credit card)</span>


Tokens per API call (`input` = full-price input, `cached` = cached input):

In [15]:
calls = pd.DataFrame(
    [t.tokens for t in client.get_turns() if t.role == "assistant"],
    columns=["input", "output", "cached"],
)
calls.loc["total"] = calls.sum()
calls

,input,output,cached
0,3781,71,0
1,4345,128,0
2,4632,141,0
total,12758,340,0


Now the **update** tool — same flow, but the LLM never sees data:

In [16]:
# Fresh client with update_dashboard callback
dashboard_sql = {"query": None, "title": None}


def on_update(data):  # needed def: querychat calls it with the filter SQL; inline alternative: none
    dashboard_sql.update(data)


client2 = qc.client(update_dashboard=on_update)
step[0] = 0
client2.on_tool_request(show_request)
client2.on_tool_result(show_result)

print("── Step 0: User sends message ──────────────────────")
print('   "Show me only cash trips over 10 miles"')

response = client2.chat("Show me only cash trips over 10 miles", echo="none")

step[0] += 1
print(f"\n── Step {step[0]}: LLM responds to user ─────────────────")
print(f"   {response}")

── Step 0: User sends message ──────────────────────
   "Show me only cash trips over 10 miles"



── Step 1: LLM requests tool ─────────────────
   Tool:      querychat_get_schema
   Arguments: {'table_name': 'taxi'}



── Step 2: Tool returns result ──────────────────
   Sent back to LLM: Table: taxi
Columns:
- VendorID (INTEGER)
  Range: 1 to 6
- tpep_pickup_datetime (TIMESTAMP)
  Range: 2002-12-31 22:59:39 to 2024-02-01 00:01:15
- tpep_dropoff_datetime (TIMESTAMP)
  Range: 2002-12-31 23:05:41 to 2024-02-02 13:56:52
- passenger_count (INTEGER)
  Range: 0 to 9
- trip_distance (FLOAT)
   Shown on screen:  title=None label='taxi' value_preview='19 columns' icon=None html=<div class="table-responsive">
  <table class="table table-sm table-hover align-middle mb-0">
    <thead class="table-light">
      <tr>
        <th scope="col">Column</th>
        <th scope="col">Type</th>
        <th scope="col">Ran



── Step 3: LLM requests tool ─────────────────
   Tool:      querychat_update_dashboard
   Arguments: {'query': 'SELECT *\nFROM taxi\nWHERE payment_type = 1 AND trip_distance > 10  -- 1 = Cash', 'table': 'taxi', 'title': 'Cash trips over 10 miles'}

── Step 4: Tool returns result ──────────────────
   Sent back to LLM: Dashboard updated. Use `query` tool to review results, if needed.
   Shown on screen:  ```sql
SELECT *
FROM taxi
WHERE payment_type = 1 AND trip_distance > 10  -- 1 = Cash
```

<button
                class="btn btn-outline-primary btn-sm float-end mt-3 querychat-update-dashboard-btn"
                data-table="taxi"
                data-query="SELECT *
FROM taxi
WHERE payment_type =



── Step 5: LLM responds to user ─────────────────
   The dashboard now shows all cash trips longer than 10 miles.


In [17]:
# This is what Shiny would do with the callback
print(f"Title: {dashboard_sql['title']}")
print()

if dashboard_sql["query"]:
    print(duckdb.sql(dashboard_sql["query"]).df().head())

Title: Cash trips over 10 miles

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2024-01-01 00:49:44   2024-01-01 01:15:47                2   
1         1  2024-01-01 00:42:05   2024-01-01 01:16:49                1   
2         2  2024-01-01 00:15:26   2024-01-01 00:33:57                1   
3         2  2024-01-01 00:36:26   2024-01-01 01:08:23                1   
4         2  2024-01-01 00:46:16   2024-01-01 01:20:10                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0          10.82           1                  N           138           181   
1          23.90           5                  N           263           265   
2          11.51           1                  N           262           171   
3          11.48           1                  N           148           244   
4          20.85           2                  N           132           239   

   payment_type  fare_amount  extra  mta_

| | `querychat_update_dashboard` | `querychat_query` |
|---|---|---|
| **Purpose** | Filter the dashboard | Answer a question |
| **LLM generates** | SQL query + title | SQL query |
| **Tool returns** | `"Dashboard updated."` | JSON data rows |
| **Rows sent to the provider?** | **No** | **Yes** |

## 4. Structured output

*Adapted from Daniel Chen, DSCI 532.*

- `chatlas`: `Chat.chat_structured()` method
- `pydantic`: data model from `BaseModel`, with optional `Field` descriptions

In [18]:
class Person(BaseModel):
    name: str
    age: int


chat = ctl.ChatOpenAI(model="gpt-5.4-nano")
chat.chat_structured(
    "My name is Susan and I'm 13 years old",
    data_model=Person,
)

Person(name='Susan', age=13)

An answer schema the model must fill for a taxi question. The model gets the column list, no rows.

In [19]:
class TaxiAnswer(BaseModel):
    sql: str = Field(description="One DuckDB SELECT query over the table taxi that answers the question")
    value: float = Field(description="Your estimate of the number the query returns")
    unit: str = Field(description="Unit of the value, e.g. dollars, miles, trips, percent")
    confidence: float = Field(description="How sure you are that the query answers the question, from 0 to 1")


schema_text = duckdb.sql("DESCRIBE taxi").df()[["column_name", "column_type"]].to_string(index=False)
print(schema_text)

          column_name column_type
             VendorID     INTEGER
 tpep_pickup_datetime   TIMESTAMP
tpep_dropoff_datetime   TIMESTAMP
      passenger_count      BIGINT
        trip_distance      DOUBLE
           RatecodeID      BIGINT
   store_and_fwd_flag     VARCHAR
         PULocationID     INTEGER
         DOLocationID     INTEGER
         payment_type      BIGINT
          fare_amount      DOUBLE
                extra      DOUBLE
              mta_tax      DOUBLE
           tip_amount      DOUBLE
         tolls_amount      DOUBLE
improvement_surcharge      DOUBLE
         total_amount      DOUBLE
 congestion_surcharge      DOUBLE
          Airport_fee      DOUBLE


In [20]:
chat = ctl.ChatOpenAI(model="gpt-5.4-nano")
answer = chat.chat_structured(
    f"Table taxi has these columns:\n{schema_text}\n\nQuestion: What is the average trip distance in miles?",
    data_model=TaxiAnswer,
)
answer

TaxiAnswer(sql='SELECT AVG(trip_distance) AS avg_trip_distance_miles\nFROM taxi;', value=0.0, unit='miles', confidence=0.6)

In [21]:
duckdb.sql(answer.sql).df()

,avg_trip_distance_miles
0,3.652169


In [22]:
chat = ctl.ChatOpenAI(model="gpt-5.4-nano")
answer = chat.chat_structured(
    f"Table taxi has these columns:\n{schema_text}\n\nQuestion: How many trips were charged the JFK airport flat rate?",
    data_model=TaxiAnswer,
)
answer

TaxiAnswer(sql='SELECT COUNT(*) AS trips\nFROM taxi\nWHERE RatecodeID = 2\n  AND (DOLocationID = 138 OR PULocationID = 138);', value=0.0, unit='trips', confidence=0.2)

In [23]:
duckdb.sql(answer.sql).df()

,trips
0,410


The code for the JFK airport flat rate, `RatecodeID` 2, is in the codebook, not in the table. The codebook is the document that says what each code means:

In [24]:
duckdb.sql("SELECT COUNT(*) AS jfk_flat_rate_trips FROM taxi WHERE RatecodeID = 2").df()

,jfk_flat_rate_trips
0,98713


## 5. RAG: codes the table cannot explain

The schema gives column names and ranges, not what the codes mean. Retrieval-augmented generation (RAG) finds the codebook paragraphs that match the question and adds them to the prompt.

The knowledge base is one `.txt` file of 6 paragraphs, one topic each. TF-IDF retrieval needs no API key or model download.

In [25]:
kb_chunks = [c.strip() for c in open("knowledge_base/taxi_glossary.txt").read().split("\n\n") if c.strip()]
kb_vectorizer = TfidfVectorizer()
kb_vectors = kb_vectorizer.fit_transform(kb_chunks)
print(f"Loaded {len(kb_chunks)} chunks → TF-IDF matrix shape: {kb_vectors.shape}")

Loaded 6 chunks → TF-IDF matrix shape: (6, 122)


In [26]:
rag_question = "What is the average tip when the passenger paid cash?"

In [27]:
q_vec = kb_vectorizer.transform([rag_question])
scores = cosine_similarity(q_vec, kb_vectors).flatten()
top_idx = np.argsort(scores)[::-1][:3]
chunks = [kb_chunks[i] for i in top_idx if scores[i] > 0]

for i in top_idx:
    print(f"{scores[i]:.3f}  {kb_chunks[i][:80]}")

0.333  payment_type codes (how the passenger paid):
1 = Credit card
2 = Cash
3 = No cha
0.243  fare_amount is the time-and-distance fare calculated by the meter.
It does not i
0.193  trip_distance is measured in miles from the taximeter.
Short trips are under 1 m


Without RAG: the question goes straight to the model.

In [28]:
response = qc.client().chat(rag_question, echo="none")
print("WITHOUT RAG:", response)

WITHOUT RAG: The average tip for cash payments (payment_type = 1) is **$4.17**, based on 2,319,046 cash-fare records.

<ul>
<li><span class="suggestion">How does the average tip compare across payment types?</span></li>
<li><span class="suggestion">What percentage of trips were paid with cash?</span></li>
<li><span class="suggestion">Show top trips by tip amount among cash payments</span></li>
</ul>


With RAG: the retrieved chunks are added to the **user message**. The tool loop is unchanged.

In [29]:
context = "\n\n".join(chunks)
augmented = f"Relevant domain context:\n{context}\n\nQuestion: {rag_question}"
print(augmented)

Relevant domain context:
payment_type codes (how the passenger paid):
1 = Credit card
2 = Cash
3 = No charge
4 = Dispute
5 = Unknown
6 = Voided trip
Use payment_type when filtering by payment method.

fare_amount is the time-and-distance fare calculated by the meter.
It does not include tips, tolls, or surcharges.
total_amount includes all charges: fare, tip, tolls, surcharges, and taxes.
tip_amount is only recorded for credit card payments. Cash tips are not captured.

trip_distance is measured in miles from the taximeter.
Short trips are under 1 mile. Long trips are over 10 miles.
Average trip distance in 2024 is approximately 3.2 miles.

Question: What is the average tip when the passenger paid cash?


In [30]:
response = qc.client().chat(augmented, echo="none")
print("WITH RAG:", response)

WITH RAG: Let me first check the schema to confirm the column names, then find the answer.

Good, I can see the schema. Now, an important note from the domain context: **tip_amount is only recorded for credit card payments. Cash tips are not captured.** Let me still run the query to confirm this in the data.

The average tip for cash payments is essentially **$0.00** (≈ $0.0023 across ~439K trips).

This is expected — as noted in the domain context, **cash tips are not captured** in the `tip_amount` field. Out of ~439,191 cash trips, only 128 have a non-zero tip (likely data entry errors), so the tip_amount field is effectively zero for cash payments.

So while passengers may well tip when paying cash, the dataset simply doesn't record those tips.

For comparison, here's the average tip by payment type:

| Payment Type | Avg Tip | Trips with Tip |
|---|---|---|
| 0 (unknown) | $1.55 | 48,799 |
| 1 (Credit Card) | $4.17 | 2,204,834 |
| 2 (Cash) | ~$0.00 | 128 |
| 3 (No Charge) | ~$0.01 

## 6. Cost per question

Prices in $ per 1M tokens, read from the providers' pricing pages on 2026-11-02:
Groq [console.groq.com/docs/models](https://console.groq.com/docs/models) (no cached-input price listed for Qwen),
OpenAI [developers.openai.com/api/docs/pricing](https://developers.openai.com/api/docs/pricing) (Standard tier).

In [31]:
PRICES = {  # model: (input, output, cached input), $ per 1M tokens, 2026-11-02
    "qwen/qwen3.8-27b": (0.80, 4.00, 0.80),
    "gpt-5.4-nano": (0.20, 1.25, 0.02),
    "gpt-5.4-mini": (0.75, 4.50, 0.075),
}

The same question through querychat on each model (one user message; several API calls each):

In [32]:
rows = []
for model, chat_client in [
    ("qwen/qwen3.8-27b", ctl.ChatGroq(model="qwen/qwen3.8-27b")),
    ("gpt-5.4-nano", ctl.ChatOpenAI(model="gpt-5.4-nano")),
    ("gpt-5.4-mini", ctl.ChatOpenAI(model="gpt-5.4-mini")),
]:
    model_client = QueryChat(taxi, "taxi", client=chat_client, tools=("update", "query")).client()
    reply = model_client.chat(question, echo="none")
    tokens = [t.tokens for t in model_client.get_turns() if t.role == "assistant"]
    rows.append({
        "model": model,
        "api_calls": len(tokens),
        "input": sum(t[0] for t in tokens),
        "output": sum(t[1] for t in tokens),
        "cached": sum(t[2] for t in tokens),
        "answer": str(reply)[:120],
    })

cost = pd.DataFrame(rows)
cost

,model,api_calls,input,output,cached,answer
0,qwen/qwen3.8-27b,3,12756,492,0,I'll help you find the average tip by payment ...
1,gpt-5.4-nano,3,1061,305,9728,Here’s the **average `tip_amount` by `payment_...
2,gpt-5.4-mini,3,1053,289,9728,Here’s the average tip amount by payment type:...


In [33]:
cost["usd_per_question"] = [
    (r.input * PRICES[r.model][0] + r.output * PRICES[r.model][1] + r.cached * PRICES[r.model][2]) / 1e6
    for r in cost.itertuples()
]
cost["questions_per_usd"] = (1 / cost["usd_per_question"]).round()
cost[["model", "api_calls", "input", "output", "cached", "usd_per_question", "questions_per_usd"]]

,model,api_calls,input,output,cached,usd_per_question,questions_per_usd
0,qwen/qwen3.8-27b,3,12756,492,0,0.012173,82.0
1,gpt-5.4-nano,3,1061,305,9728,0.000788,1269.0
2,gpt-5.4-mini,3,1053,289,9728,0.002820,355.0
